# 10 — Locked evaluation

This notebook measures active-fault detection, pre-impact early warning,
48-hour prompt detection, operational incident burden, delay and localisation
after the configuration is frozen. Development is the default. Holdout can be
opened only after Notebook 07 qualifies early warning and both
`EVALUATION_PARTITION=holdout` and `OPEN_HOLDOUT=1` are set.

Before any holdout truth is read, an append-only ledger records the selected
configuration. A different configuration is then refused. The notebook also
replays development through the same frozen scoring function used for holdout
and requires byte-identical scores. Evaluation cannot change any model or
policy value.


## 1. Select the evaluation partition deliberately


In [ ]:
from pathlib import Path
import os
import sys

if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")


def find_repository(start=Path.cwd()):
    """Find the checked-out repository when Jupyter starts in any subfolder."""
    override = os.getenv("TELCO_PROJECT_ROOT")
    if override:
        candidates = [Path(override).expanduser().resolve()]
    else:
        start = start.resolve()
        candidates = [start, *start.parents]
        if "google.colab" in sys.modules:
            candidates += [
                Path("/content/drive/MyDrive/anomaly_detection"),
                Path("/content/drive/MyDrive/telco-anomaly-detection"),
            ]
    for candidate in candidates:
        if (candidate / "pyproject.toml").is_file() and (candidate / "configs").is_dir():
            return candidate.resolve()
    raise FileNotFoundError(
        "Open this notebook from the cloned repository, or set TELCO_PROJECT_ROOT."
    )


PROJECT_ROOT = find_repository()
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

from datetime import datetime, timezone
import tempfile

import joblib
import pandas as pd
from IPython.display import display

from telco_anomaly.detectors import (
    alerts_from_score_file,
    materialize_measurement_features,
    materialize_wide_partition,
    partition_exposure,
    score_partition_file,
)
from telco_anomaly.evaluation import (
    evaluate_cases, form_cases, scoreable_exposure, wilson_interval,
)
from telco_anomaly.selection import qualify_localisation
from telco_anomaly.io import (
    authorise_holdout,
    file_sha256,
    immutable_output_directory,
    read_json,
    require_same,
    resolve_data_root,
    write_json,
)

DATA_ROOT = resolve_data_root()
PARTITION = os.getenv("EVALUATION_PARTITION", "development").lower()
OPEN_HOLDOUT = os.getenv("OPEN_HOLDOUT", "0") == "1"
if PARTITION not in {"development", "holdout"}:
    raise ValueError("EVALUATION_PARTITION must be development or holdout")
if PARTITION == "holdout" and not OPEN_HOLDOUT:
    raise PermissionError(
        "Holdout remains sealed. Set OPEN_HOLDOUT=1 only after selection is frozen."
    )

CORE_RUN_ID = os.getenv(
    "TELCO_CORE_RUN_ID", os.getenv("PON_CORE_RUN_ID", "synthetic_pon_core_v2")
)
FEATURE_RUN_ID = os.getenv(
    "TELCO_FEATURE_RUN_ID", os.getenv("PON_FEATURE_RUN_ID", "synthetic_pon_features_v5")
)
MODEL_RUN_ID = os.getenv(
    "TELCO_MODEL_RUN_ID", os.getenv("PON_MODEL_RUN_ID", "synthetic_pon_models_v9")
)
TRUTH_RUN_ID = os.getenv("PON_TRUTH_RUN_ID", "synthetic_pon_truth_v3")
SELECTION_RUN_ID = os.getenv("PON_SELECTION_RUN_ID", "synthetic_pon_selection_v11")
INCIDENT_RUN_ID = os.getenv("PON_INCIDENT_RUN_ID", "synthetic_pon_incidents_v11")
EVALUATION_RUN_ID = os.getenv(
    "PON_EVALUATION_RUN_ID", f"synthetic_pon_{PARTITION}_v11"
)

RUN_ROOT = DATA_ROOT / "core" / "synthetic_pon" / CORE_RUN_ID
CORE_ROOT = RUN_ROOT / "SPEC-CORE"
FEATURE_ROOT = DATA_ROOT / "features" / "synthetic_pon" / FEATURE_RUN_ID
MODEL_ROOT = DATA_ROOT / "models" / "synthetic_pon" / MODEL_RUN_ID
TRUTH_ROOT = DATA_ROOT / "evaluation" / "synthetic_pon" / TRUTH_RUN_ID
SELECTION_ROOT = DATA_ROOT / "selection" / "synthetic_pon" / SELECTION_RUN_ID
INCIDENT_ROOT = DATA_ROOT / "incidents" / "synthetic_pon" / INCIDENT_RUN_ID
OUTPUT_ROOT = DATA_ROOT / "results" / "synthetic_pon" / EVALUATION_RUN_ID
SCRATCH_PARENT = Path(os.getenv(
    "TELCO_WORK_ROOT",
    "/content" if "google.colab" in sys.modules else tempfile.gettempdir(),
))
SCRATCH_PARENT.mkdir(parents=True, exist_ok=True)

selected_path = SELECTION_ROOT / "selected_configuration.json"
if not selected_path.exists():
    raise RuntimeError("No selected configuration passed Notebook 07's development gates")
configuration = read_json(selected_path)
if PARTITION == "holdout" and not configuration.get("holdout_ready", False):
    raise PermissionError(
        "Holdout remains sealed because Notebook 07 did not qualify early warning."
    )
core_manifest = read_json(CORE_ROOT / "manifest.json")
truth_manifest_path = TRUTH_ROOT / "truth_manifest.json"
truth_manifest = read_json(truth_manifest_path)
model_manifest_path = MODEL_ROOT / "model_manifest.json"
model_manifest = read_json(model_manifest_path)
feature_manifest_path = FEATURE_ROOT / "feature_manifest.json"
feature_manifest = read_json(feature_manifest_path)
resolved_policy_path = MODEL_ROOT / "resolved_policy.json"
resolved_policy = read_json(resolved_policy_path)

require_same(
    configuration,
    model_manifest_sha256=file_sha256(model_manifest_path),
    resolved_policy_sha256=model_manifest["resolved_policy_sha256"],
    development_truth_manifest_sha256=file_sha256(truth_manifest_path),
    evaluation_module_sha256=file_sha256(
        PROJECT_ROOT / "src" / "telco_anomaly" / "evaluation.py"
    ),
    selection_module_sha256=file_sha256(
        PROJECT_ROOT / "src" / "telco_anomaly" / "selection.py"
    ),
)
require_same(
    model_manifest,
    feature_manifest_sha256=file_sha256(feature_manifest_path),
    core_fingerprint=core_manifest["fingerprint"],
)
require_same(
    truth_manifest, model_core_fingerprint=core_manifest["fingerprint"]
)
# Holdout features use the frozen manifest, not the current features.yml.
require_same(
    feature_manifest,
    core_fingerprint=core_manifest["fingerprint"],
    time_partitions_sha256=file_sha256(
        RUN_ROOT / "SPLITS" / "time_partitions.parquet"
    ),
    features_module_sha256=file_sha256(
        PROJECT_ROOT / "src" / "telco_anomaly" / "features.py"
    ),
)
if file_sha256(
    PROJECT_ROOT / "src" / "telco_anomaly" / "detectors.py"
) != model_manifest["detectors_module_sha256"]:
    raise ValueError("Detector code changed after model fitting")
if file_sha256(
    PROJECT_ROOT / "configs" / "topology.yml"
) != model_manifest["topology_config_sha256"]:
    raise ValueError("Topology policy changed after model fitting")
if file_sha256(resolved_policy_path) != model_manifest["resolved_policy_sha256"]:
    raise ValueError("The frozen model policy no longer matches its manifest")

POLICY = resolved_policy["alert_policy"]
catalogue = pd.read_parquet(CORE_ROOT / "metric_catalogue.parquet")
topology_path = CORE_ROOT / "topology_memberships.parquet"
topology = pd.read_parquet(topology_path) if topology_path.exists() else pd.DataFrame()
reference_path = MODEL_ROOT / "topology_reference.parquet"
topology_reference = (
    pd.read_parquet(reference_path) if reference_path.exists() else pd.DataFrame()
)
truth_directory = TRUTH_ROOT / (
    "holdout_locked" if PARTITION == "holdout" else "development"
)

receipt = {
    "opened_at_utc": datetime.now(timezone.utc).isoformat(),
    "partition": PARTITION,
    "holdout_authorised": bool(PARTITION == "holdout" and OPEN_HOLDOUT),
    "selected_configuration_sha256": file_sha256(selected_path),
    "core_fingerprint": core_manifest["fingerprint"],
    "truth_manifest_sha256": file_sha256(truth_manifest_path),
    "model_manifest_sha256": file_sha256(model_manifest_path),
    "feature_manifest_sha256": file_sha256(feature_manifest_path),
    "resolved_policy_sha256": file_sha256(resolved_policy_path),
    "evaluation_module_sha256": file_sha256(
        PROJECT_ROOT / "src" / "telco_anomaly" / "evaluation.py"
    ),
    "selection_module_sha256": file_sha256(
        PROJECT_ROOT / "src" / "telco_anomaly" / "selection.py"
    ),
    "model_run_id": MODEL_RUN_ID,
    "selection_run_id": SELECTION_RUN_ID,
}
if PARTITION == "holdout":
    ledger = TRUTH_ROOT.with_name(f"{TRUTH_ROOT.name}_holdout_openings.jsonl")
    opening_number = authorise_holdout(ledger, receipt)
    print(f"Authorised holdout opening {opening_number} for this frozen configuration")

display(pd.Series(receipt, name="evaluation receipt").to_frame())


## 2. Reconstruct frozen incidents for the requested partition


In [ ]:
def alerts_and_cases(score_path):
    frames = [
        alerts_from_score_file(
            score_path,
            channel,
            configuration["thresholds"][channel],
            min_consecutive=configuration["persistence_observations"][channel],
            recovery_consecutive=configuration["recovery_observations"],
            cadence_seconds=configuration["cadence_seconds"],
            recovery_threshold_fraction=(
                configuration["recovery_threshold_fraction"]
            ),
        )
        for channel in configuration["channels"]
    ]
    alerts = pd.concat(frames, ignore_index=True).sort_values("alert_start")
    alerts = alerts.reset_index(drop=True)
    alerts["alert_id"] = [f"A-{number:09d}" for number in range(1, len(alerts) + 1)]
    cases, members = form_cases(
        alerts,
        topology,
        gap_seconds=configuration["incident_quiet_period_seconds"],
        thresholds=configuration["thresholds"],
        shared_scope_models=("group_common_mode",),
    )
    return alerts, cases, members


if PARTITION == "development":
    score_path = MODEL_ROOT / model_manifest["score_files"]["development"]
    calendar_exposure = partition_exposure(
        score_path, "entity_day", model_manifest["cadence_seconds"]
    )
    exposure = scoreable_exposure(
        score_path, configuration["channels"], "entity_day",
        model_manifest["cadence_seconds"],
    )
    alerts, cases, members = alerts_and_cases(score_path)
else:
    bundle = joblib.load(MODEL_ROOT / "residual_bundle.joblib")
    contextual_bundle = bundle.get("contextual_isolation_bundle")
    feature_lookback_seconds = max(
        *feature_manifest["history_windows_seconds"].values(),
        *feature_manifest["lag_windows_seconds"].values(),
        *feature_manifest["activity_windows_seconds"].values(),
        *feature_manifest.get("seasonal_periods", {}).values(),
    )

    # Prove that the reusable scorer reproduces the frozen development file
    # before it is allowed to touch holdout features.
    with tempfile.TemporaryDirectory(
        dir=SCRATCH_PARENT, prefix="telco-scoring-replay-"
    ) as replay_name:
        replay_workspace = Path(replay_name)
        replay_path = replay_workspace / "development_scores.parquet"
        score_partition_file(
            bundle,
            FEATURE_ROOT / feature_manifest["partitions"]["development"]["features"],
            replay_path,
            replay_workspace / "work",
            cadence_seconds=model_manifest["cadence_seconds"],
            dispersion_window_seconds=model_manifest["dispersion_window_seconds"],
            resolved_policy=resolved_policy,
            topology=topology,
            topology_reference=topology_reference,
            contextual_isolation_bundle=contextual_bundle,
        )
        frozen_development = (
            MODEL_ROOT / model_manifest["score_files"]["development"]
        )
        if file_sha256(replay_path) != file_sha256(frozen_development):
            raise ValueError(
                "The shared scorer did not reproduce frozen development scores"
            )
    print("PASS — holdout will use the development scoring path exactly")

    with tempfile.TemporaryDirectory(
        dir=SCRATCH_PARENT, prefix="telco-holdout-"
    ) as temporary_name:
        temporary = Path(temporary_name)
        wide = temporary / "holdout_wide.parquet"
        features = temporary / "holdout_features.parquet"
        score_path = temporary / "holdout_scores.parquet"
        split = materialize_wide_partition(
            CORE_ROOT,
            RUN_ROOT / "SPLITS",
            "holdout",
            catalogue,
            wide,
            lookback_seconds=feature_lookback_seconds,
            memory_limit=os.getenv("DUCKDB_MEMORY_LIMIT", "2GB"),
            threads=int(os.getenv("DUCKDB_THREADS", "2")),
        )
        materialize_measurement_features(
            wide,
            catalogue,
            features,
            gap_tolerance=feature_manifest["gap_tolerance"],
            seasonal_periods=feature_manifest.get("seasonal_periods", {}),
            history_windows_seconds=feature_manifest["history_windows_seconds"],
            lag_windows_seconds=feature_manifest["lag_windows_seconds"],
            activity_windows_seconds=feature_manifest["activity_windows_seconds"],
            history_metric_ids=feature_manifest["history_metric_ids"],
            lag_metric_ids=feature_manifest["lag_metric_ids"],
            activity_metric_ids=feature_manifest["activity_metric_ids"],
            minimum_window_fraction=feature_manifest["minimum_window_fraction"],
            score_start=split["score_start"],
            score_end=split["score_end"],
            progress_every=25,
        )
        score_partition_file(
            bundle,
            features,
            score_path,
            temporary / "score_work",
            cadence_seconds=model_manifest["cadence_seconds"],
            dispersion_window_seconds=model_manifest["dispersion_window_seconds"],
            resolved_policy=resolved_policy,
            topology=topology,
            topology_reference=topology_reference,
            contextual_isolation_bundle=contextual_bundle,
        )
        calendar_exposure = partition_exposure(
            score_path, "entity_day", model_manifest["cadence_seconds"]
        )
        exposure = scoreable_exposure(
            score_path, configuration["channels"], "entity_day",
            model_manifest["cadence_seconds"],
        )
        alerts, cases, members = alerts_and_cases(score_path)


## 3. Open truth only after all decisions are frozen


In [ ]:
truth_folder = truth_directory.name
for name in ("fault_events.parquet", "fault_entity_intervals.parquet"):
    relative = f"{truth_folder}/{name}"
    if file_sha256(truth_directory / name) != truth_manifest["file_sha256"].get(relative):
        raise ValueError(f"Evaluation truth file changed: {relative}")
events = pd.read_parquet(truth_directory / "fault_events.parquet")
intervals = pd.read_parquet(truth_directory / "fault_entity_intervals.parquet")

# Active-interval recall measures anomaly detection. Pre-impact recall from
# this result measures early warning; a second pass measures the 48-hour SLA.
result = evaluate_cases(
    cases,
    members,
    events,
    intervals,
    exposure_value=exposure,
    exposure_unit="entity_day",
    decision_horizon_seconds=None,
    topology_memberships=topology,
    confidence_level=POLICY["workload"]["confidence_level"],
)
prompt_result = evaluate_cases(
    cases,
    members,
    events,
    intervals,
    exposure_value=exposure,
    exposure_unit="entity_day",
    decision_horizon_seconds=configuration["prompt_detection_horizon_seconds"],
    topology_memberships=topology,
    confidence_level=POLICY["workload"]["confidence_level"],
)

affected_counts = (
    intervals.assign(fault_id=intervals["fault_id"].astype(str))
    .groupby("fault_id")["entity_id"].nunique()
)
multi_fault_ids = set(affected_counts.loc[affected_counts.gt(1)].index)
multi_entity = result["fault_results"].loc[
    result["fault_results"]["fault_id"].astype(str).isin(multi_fault_ids)
]
multi_successes = int(
    (multi_entity["detected"] & multi_entity["equivalent_scope"]).sum()
)
joint_low, joint_high = wilson_interval(
    multi_successes, len(multi_entity),
    POLICY["workload"]["confidence_level"],
)
localisation_row = pd.Series({
    "multi_entity_faults": len(multi_entity),
    "multi_entity_detected_and_localised": multi_successes,
    "multi_entity_joint_detection_and_localisation_recall": (
        multi_successes / len(multi_entity) if len(multi_entity) else float("nan")
    ),
    "multi_entity_joint_detection_and_localisation_recall_ci_low": joint_low,
    "multi_entity_joint_detection_and_localisation_recall_ci_high": joint_high,
})
localisation_assessment = qualify_localisation(
    localisation_row,
    minimum_multi_entity_faults=POLICY["selection"]["minimum_multi_entity_faults_for_localisation_claim"],
    minimum_joint_recall_ci_low=POLICY["selection"].get(
        "minimum_joint_localisation_recall_ci_low"
    ),
)

metrics = result["metrics"].copy()
prompt_recall = prompt_result["metrics"].loc[
    prompt_result["metrics"]["metric"].eq("event_recall")
].copy()
prompt_recall["metric"] = "prompt_event_recall"
metrics = pd.concat([metrics, prompt_recall], ignore_index=True)
metrics["display_metric"] = metrics["metric"].replace({
    "case_precision": "incident_precision",
    "false_cases_per_entity_day": "false_incidents_per_entity_day",
    "total_cases_per_entity_day": "total_incidents_per_entity_day",
})
primary_metrics = set(POLICY["evaluation"]["primary_metrics"]) | {
    "preimpact_event_recall", "prompt_event_recall",
}
primary = metrics.loc[metrics["display_metric"].isin(primary_metrics)]
display(primary[["display_metric", "value", "ci_low", "ci_high", "numerator", "denominator"]])
display(result["fault_type_results"])
display(pd.Series(localisation_assessment, name="localisation evidence").to_frame())


## 4. Publish the immutable result and limitations


In [ ]:
small_fault_types = result["fault_type_results"].loc[
    result["fault_type_results"]["reporting_status"].ne("estimable"), "fault_type"
].tolist()
evaluation_manifest = {
    **receipt,
    "configuration_status": configuration["status"],
    "model_manifest_sha256": receipt["model_manifest_sha256"],
    "feature_manifest_sha256": receipt["feature_manifest_sha256"],
    "resolved_policy_sha256": receipt["resolved_policy_sha256"],
    "scoreable_faults": events["fault_id"].nunique(),
    "calendar_exposure_entity_days": calendar_exposure,
    "scoreable_exposure_entity_days": exposure,
    "score_availability": (
        exposure / calendar_exposure if calendar_exposure else None
    ),
    "incidents": len(cases),
    "detection_qualification": configuration["status"],
    "evaluation_estimands": {
        "event_recall": "first detection during the active fault interval",
        "preimpact_event_recall": "first detection after observable evidence and before impact",
        "prompt_event_recall": (
            f"first detection within {configuration['prompt_detection_horizon_seconds']} seconds"
        ),
    },
    "early_warning_qualification": configuration["early_warning_qualification"],
    "localisation_qualification": localisation_assessment,
    "small_fault_types_descriptive_only": small_fault_types,
    "model_changes_allowed": False,
    "limitations": [
        "Synthetic PON results validate injected mechanisms, not operator prevalence.",
        "Sparse fault-type rows are descriptive rather than stable estimates.",
        "Localisation is a topology-scope estimate, not causal root-cause proof.",
        "Localisation remains separate and is not implied by detection selection.",
    ],
}

if OUTPUT_ROOT.exists():
    previous_receipt = read_json(OUTPUT_ROOT / "evaluation_receipt.json")
    require_same(
        previous_receipt,
        partition=receipt["partition"],
        selected_configuration_sha256=receipt["selected_configuration_sha256"],
        core_fingerprint=receipt["core_fingerprint"],
        truth_manifest_sha256=receipt["truth_manifest_sha256"],
        model_manifest_sha256=receipt["model_manifest_sha256"],
        feature_manifest_sha256=receipt["feature_manifest_sha256"],
        resolved_policy_sha256=receipt["resolved_policy_sha256"],
    )
    print("Using existing immutable evaluation:", OUTPUT_ROOT)
else:
    with immutable_output_directory(OUTPUT_ROOT) as output:
        metrics.to_parquet(output / "metrics.parquet", index=False)
        result["fault_type_results"].to_parquet(output / "fault_type_results.parquet", index=False)
        result["domain_type_results"].to_parquet(output / "domain_type_results.parquet", index=False)
        result["localisation_results"].to_parquet(output / "localisation_results.parquet", index=False)
        result["fault_results"].to_parquet(output / "fault_results.parquet", index=False)
        write_json(output / "localisation_assessment.json", localisation_assessment)
        write_json(output / "evaluation_receipt.json", receipt)
        write_json(output / "evaluation_manifest.json", evaluation_manifest)
    print("Saved locked evaluation:", OUTPUT_ROOT)
print("No modelling decision may be changed from this notebook's results.")
print("Next: 11_PUBLIC_DATASET_VALIDATION.ipynb")
